In [ ]:
import pandas as pd
import numpy as np
import os
import ast 

anova_kaggle = anova_mendeley = {}
chi2_kaggle = chi2_mendeley = {}
mi_kaggle = mi_mendeley = {}
vt_kaggle = vt_mendeley = {}
for file in (os.listdir('../results/phase_2')):
    if 'final_summary' in file:
        df = pd.read_csv(f'../results/phase_2/{file}')
        kaggle_features = df.loc[df['Dataset'] == 'Kaggle', ["Method", "Model", "Features_Used"]]
        for index, row in kaggle_features.iterrows():
    
            method = row["Method"]
            model_name = row["Model"]
            feature_list = ast.literal_eval(row["Features_Used"])
            
            match method:
                case "Anova":
                    anova_kaggle[model_name] = feature_list
                case "Variance Threshold":
                    vt_kaggle[model_name] = feature_list
                case "Mutual Info":
                    mi_kaggle[model_name] = feature_list
                case "Chi Square":
                    chi2_kaggle[model_name] = feature_list
        
        mendeley_features = df.loc[df['Dataset'] == 'Mendeley', ["Method", "Model", "Features_Used"]]
        for index, row in mendeley_features.iterrows():
            method = row["Method"]
            model_name = row["Model"]
            feature_list = ast.literal_eval(row["Features_Used"])
            
            match method:
                case "Anova":
                    anova_mendeley[model_name] = feature_list
                case "Variance Threshold":
                    vt_mendeley[model_name] = feature_list
                case "Mutual Info":
                    mi_mendeley[model_name] = feature_list
                case "Chi Square":
                    chi2_mendeley[model_name] = feature_list


In [ ]:
from data_preprocessing import create_train_test_val_sets, get_processed_df

#Create test train splits
x_mendeley, y_mendeley = get_processed_df(r"..\data\raw\Mendeley Dataset.csv")
x_kaggle, y_kaggle= get_processed_df(r"..\data\raw\dataset_phishing.csv")

mendeley_sets = create_train_test_val_sets(x_mendeley,y_mendeley, label_col="Label", test_size=0.2, n_splits=5)
kaggle_sets = create_train_test_val_sets(x_kaggle,y_kaggle, label_col="Label", test_size=0.2, n_splits=5)

In [ ]:
import joblib 

#import phase 1 models
xgb_mendeley = joblib.load('./models/phase_1/xgboost_mendeley_no_fs.joblib')
xgb_kaggle = joblib.load('./models/phase_1/xgboost_kaggle_no_fs.joblib')
logreg_mendeley = joblib.load('./models/phase_1/logreg_mendeley_no_fs.joblib')
logreg_kaggle = joblib.load('./models/phase_1/logreg_kaggle_no_fs.joblib')
rf_mendeley = joblib.load('./models/phase_1/rf_mendeley_no_fs.joblib')
rf_kaggle = joblib.load('./models/phase_1/rf_kaggle_no_fs.joblib')

In [ ]:
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

def remove_redundant_features(dataset, model, dataset_name, model_name, method, selected_features):
    print(f'Redundnacy Calculation for {model_name} using {method} for {dataset_name}')
    print(f'Initial feature counts {len(selected_features)}')

    X_selected = dataset['x_train'][selected_features]
    y = dataset['y_train']

    #normalized importance scores
    mi_scores = mutual_info_classif(X_selected, y, discrete_features='auto')
    importance = pd.Series(mi_scores, index=X_selected.columns)
    importance = importance / importance.max()

    #correlation matrix - to get redundancy
    corr_matrix = X_selected.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    #remove the highly redundant pairs
    candidate_pairs = [
        (col, row)
        for col in upper.columns
        for row in upper.index
        if upper.loc[row, col] > 0.7
    ]

    mi_cache = {}
    to_remove = set()

    for f1, f2 in candidate_pairs:

        if f1 in to_remove or f2 in to_remove:
            continue

        key = tuple(sorted([f1, f2]))

        if key in mi_cache:
            mi = mi_cache[key]
        else:
            mi = mutual_info_regression(
                X_selected[[f1]], X_selected[f2]
            )[0]
            mi_cache[key] = mi

        if mi > 0.6:
            # Drop less important feature
            if importance[f1] >= importance[f2]:
                to_remove.add(f2)
            else:
                to_remove.add(f1)

    final_features = [f for f in selected_features if f not in to_remove]

    print(f"Removed {len(to_remove)} features")
    print(f"Final feature count: {len(final_features)}\n")

    return final_features



In [ ]:
#Evaluate and save 
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone

def evaluate_model(dataset, model, selected_features):
    
    X_train = pd.concat([dataset["x_train"], dataset["x_val"]])[selected_features]
    X_test = dataset["x_test"][selected_features]
    y_train = pd.concat([dataset["y_train"], dataset["y_val"]])
    y_test = dataset["y_test"]

    model_clone = clone(model)
    model_clone.fit(X_train, y_train)

    y_pred = model_clone.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)

    return f1, prec, rec

In [ ]:
#Run Mendeley
feature_sets_mendeley = {
    "Anova": anova_mendeley,
    "Variance Threshold": vt_mendeley,
    "Chi Square": chi2_mendeley,
    "Mutual Info": mi_mendeley
}

models_mendeley = {
    "XGBoost": xgb_mendeley,
    "RF": rf_mendeley,
    "LogReg": logreg_mendeley
}

results = {}

for model_name, model in models_mendeley.items():
    results[model_name] = {}

    for method, feature_dict in feature_sets_mendeley.items():

        selected_features = feature_dict[model_name]

        final_features = remove_redundant_features(
            mendeley_sets,
            model,
            "Mendeley",
            model_name,
            method,
            selected_features
        )

        results[model_name][method] = final_features

In [ ]:
final_results = []

for model_name, model in models_mendeley.items():
    for method in feature_sets_mendeley.keys():

        print(f"\nEvaluating {model_name} - {method}")

        selected_features = results[model_name][method]

        f1, prec, rec = evaluate_model(mendeley_sets, model, selected_features)

        print(f"{model_name} | {method} - F1: {f1:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

        final_results.append({
            "dataset": "Mendeley",
            "model": model_name,
            "method": method,
            "num_features": len(selected_features),
            "f1": f1,
            "precision": prec,
            "recall": rec,
            "features": selected_features
        })

In [ ]:
print(logreg_mendeley)

In [ ]:
#Run Kaggle
feature_sets_kaggle = {
    "Anova": anova_kaggle,
    "Variance Threshold": vt_kaggle,
    "Chi Square": chi2_kaggle,
    "Mutual Info": mi_kaggle
}

models_kaggle = {
    "XGBoost": xgb_kaggle,
    "RF": rf_kaggle,
    "LogReg": logreg_kaggle
}

results = {}

for model_name, model in models_kaggle.items():
    results[model_name] = {}

    for method, feature_dict in feature_sets_kaggle.items():

        selected_features = feature_dict[model_name]

        final_features = remove_redundant_features(
            mendeley_sets,
            model,
            "Mendeley",
            model_name,
            method,
            selected_features
        )

        results[model_name][method] = final_features